# Calcul de l'enveloppe convexe du système Fe-Cu-Co-Ni 
Données extraites de Materials Project

In [39]:
# Paquet necessaire : numpy, pandas, scipy, convexhull
import numpy as np
import sys
import pandas as pd
from convexhull import cvhull
import time

import time
from pymatgen.analysis.phase_diagram import PhaseDiagram
from pymatgen.entries.computed_entries import ComputedEntry

In [40]:
df = pd.read_csv("data_test/data3.csv")
df

,id,formule,Fe,Cu,Co,Ni,somme,E
0,mp-1245108,Fe,1,0,0,0,1,-8.19
1,mp-1271198,Fe,1,0,0,0,1,-8.42
2,mp-13,Fe,1,0,0,0,1,-8.47
3,mp-1194030,Fe,1,0,0,0,1,-8.29
4,mp-1271068,Fe,1,0,0,0,1,-8.45
...,...,...,...,...,...,...,...,...
78,mp-1225698,Cu₃Ni,0,3,0,1,4,-4.47
79,mp-1225695,CuNi,0,1,0,1,2,-4.87
80,mp-1225687,CuNi,0,1,0,1,2,-4.92
81,mp-1184069,CuNi,0,1,0,1,2,-4.91


In [41]:
energie = df.loc[:,'E']
enerige = np.array(energie,'float64')

nom = [x.strip() for x in df.loc[:,'id']]
nom = list(nom)

element = ['Fe','Cu','Co','Ni']
c = len(element)
compo = df.loc[:,element]
compo = np.array(compo,'float64')

energie.shape, len(nom), compo.shape

((83,), 83, (83, 4))

In [42]:
# Si besoin, on transforme les nombres d'élément en compostion
compo = [[y/x.sum() for y in x] for x in compo]
compo = np.array(compo,'float64')
compo

array([[1.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        ],
       [1.        , 0.        , 0.        , 0.        ],
       [0.        , 0.        , 1.        , 0.        ],
       [0.        , 0.        , 1.        , 0.        ],
       [0.        , 0.        , 1.        , 0.        ],
       [0.        , 0.        , 1.        , 0.        ],
       [0.        , 0.        , 1.        , 0.        ],
       [0.        , 0.        , 1.        , 0.        ],
       [0.        , 0.        , 1.        , 0.        ],
       [0.        , 0.        ,

In [44]:
test_c = np.array([
    [1,0,0],
    [0,1,0],
    [0,0,1],
    [3/4,0,1/4],
    [1/9,0,8/9],
    [0,8/9,1/9]])
test_e = [0,0,0,-0.021,-0.006,-0.001]

In [46]:
def getMatrix(equi, composition, energy) :
    m = equi.shape[0]
    matrix = np.zeros((m+1,m+1))
    j = 0
    for i in equi :
        matrix[j,:-2] = composition[i,:-1]
        matrix[j,-2] = energy[i]
        matrix[j,-1] = 1
        j+=1
    matrix[-1,-2] = 1
    return matrix

def getHP(matrix) :
    m = matrix.shape[1] - 1
    return np.linalg.solve(matrix, np.concatenate([np.zeros((m)),[1]]))

def fHP(compo, hp) :
    return -(np.dot(compo, hp[:-2]) + hp[-1])/hp[-2]

def isValidate(matrix, tol=1e-12) :
    points = matrix[:-1,:-1]
    ref = points[0]  # Choisir un point de référence  
    directions = points[1:] - ref # Construire les vecteurs directionnels : P_i - P0
    rang = np.linalg.matrix_rank(directions) # Calcul du rang
    flagRang = rang == (len(ref) - 1)

    # Check if diverge
    for i in range(points.shape[0]-1) :
        if points[:,i].std() < tol :
            return False
    return  flagRang

In [47]:
composition = compo.copy()
energy = energie.copy()


start = time.time()

N,m = composition.shape
indexList = np.arange(m)

equilibrium = np.zeros((1,m),dtype='int')

# Look for the more stable reference
ref = np.array([None]*m)
for i in range(N) :
    x = composition[i,:]
    if (x==1).any() :
        j = int((indexList*x).sum())
        if ref[j] == None or ref[j]>energy[i] :
            ref[j] = energy[i]
            equilibrium[0,j] = i
if (ref==None).any() : # Check il there is a compound for each reference
    sys.exit()

# Loop to find each facette
no_change_counter = 0
max_no_change = 50  # nombre d'itérations consécutives sans ajout

i = 0
done = 0
matrix = getMatrix(equilibrium[0], composition, energy)
hPlan = np.array([getHP(matrix)])
equilibrium[0].sort()
testEq = equilibrium.copy()
while done < 1 :
    #print(equilibrium[i])
     
    distance = energy - fHP(composition[:,:-1], hPlan[i])
    index = np.argmin(distance)
    #print (index, distance[index], end=' ')
    
    if distance[index] > -1e-12 :
        i += 1
        no_change_counter = 0
    else :
        no_change_counter +=1
        for j in range(equilibrium.shape[1]):
            new = np.concatenate((equilibrium[i,:j], equilibrium[i,j+1:],[index]))
            new.sort()
            if (new == testEq).all(1).any() : # already in equilibrium
                continue
            testEq = np.concatenate([testEq,[new]])
            matrix = getMatrix(new,composition,energy)
            if isValidate(matrix) :
                try :
                    hp = np.linalg.solve(matrix, np.concatenate([np.zeros((m)),[1]]))
                    equilibrium = np.concatenate([equilibrium,[new]])
                    hPlan = np.concatenate([hPlan,[hp]])
                except :
                    pass
        equilibrium = np.delete(equilibrium, i, axis=0)
        hPlan = np.delete(hPlan, i, axis=0)
    done = i/len(equilibrium)

end = time.time()

print(f"Temps d'exécution : {end - start:.10f} secondes")

Temps d'exécution : 0.0184974670 secondes


In [48]:
equilibrium

array([[21, 29, 55, 68],
       [29, 40, 55, 68],
       [13, 29, 40, 68],
       [29, 38, 40, 55],
       [29, 38, 55, 58],
       [29, 35, 38, 58],
       [ 2, 29, 35, 58]])

In [49]:
start = time.time()

hull = cvhull(nom, composition, energy)

end = time.time()

print(f"Temps d'exécution : {end - start:.10f} secondes")

Temps d'exécution : 0.0052177906 secondes


In [50]:
from pymatgen.core import Composition

In [51]:
# Supposons que 'compositions' est une liste de compositions
# et 'energies' est la liste correspondante des énergies de formation
elements = ["A", "B", "C","D"]
start_time = time.time()
# Conversion en objets Composition
entries = []
for xi, ener in zip(composition, energy):
    comp_dict = {el: frac for el, frac in zip(elements, xi) if frac > 1e-6}
    comp = Composition(comp_dict)
    entries.append(ComputedEntry(comp, ener))


pd = PhaseDiagram(entries)
end_time = time.time()

print(f"Temps d'exécution : {end_time - start_time} secondes")

Temps d'exécution : 0.045629262924194336 secondes


In [58]:
pd.all_entries

[None ComputedEntry - C1           (C)
 Energy (Uncorrected)     = -6.9800   eV (-6.9800  eV/atom)
 Correction               = 0.0000    eV (0.0000   eV/atom)
 Energy (Final)           = -6.9800   eV (-6.9800  eV/atom)
 Energy Adjustments:
   None
 Parameters:
 Data:,
 None ComputedEntry - C1           (C)
 Energy (Uncorrected)     = -7.1000   eV (-7.1000  eV/atom)
 Correction               = 0.0000    eV (0.0000   eV/atom)
 Energy (Final)           = -7.1000   eV (-7.1000  eV/atom)
 Energy Adjustments:
   None
 Parameters:
 Data:,
 None ComputedEntry - C1           (C)
 Energy (Uncorrected)     = -6.9900   eV (-6.9900  eV/atom)
 Correction               = 0.0000    eV (0.0000   eV/atom)
 Energy (Final)           = -6.9900   eV (-6.9900  eV/atom)
 Energy Adjustments:
   None
 Parameters:
 Data:,
 None ComputedEntry - C1           (C)
 Energy (Uncorrected)     = -7.1100   eV (-7.1100  eV/atom)
 Correction               = 0.0000    eV (0.0000   eV/atom)
 Energy (Final)           = -7.110

In [10]:
hull.stable

array([ 2, 13, 21, 29, 35, 38, 40, 55, 58, 68], dtype=int32)

In [10]:
hull.nom

['mp-1245108',
 'mp-1271198',
 'mp-13',
 'mp-1194030',
 'mp-1271068',
 'mp-1271128',
 'mp-1096950',
 'mp-136',
 'mp-568345',
 'mp-150',
 'mp-1193227',
 'mp-1271142',
 'mp-669382',
 'mp-54',
 'mp-1183710',
 'mp-102',
 'mp-1072089',
 'mp-1014111',
 'mp-1094136',
 'mp-1246134',
 'mp-10257',
 'mp-23',
 'mp-1008728',
 'mp-1056079',
 'mp-1059259',
 'mp-1010136',
 'mp-1120774',
 'mp-989695',
 'mp-989782',
 'mp-30',
 'mp-998890',
 'mp-1084832',
 'mp-641526',
 'mp-601842',
 'mp-1080603',
 'mp-18695',
 'mp-1225083',
 'mp-1007852',
 'mp-601820',
 'mp-1225016',
 'mp-2090',
 'mp-601848',
 'mp-1096987',
 'mp-1072076',
 'mp-1225009',
 'mp-1224780',
 'mp-1007862',
 'mp-1224827',
 'mp-999189',
 'mp-1007853',
 'mp-1224697',
 'mp-1077745',
 'mp-1079337',
 'mp-1007855',
 'mp-1225078',
 'mp-1418',
 'mp-1007854',
 'mp-1224965',
 'mp-2213',
 'mp-1224976',
 'mp-1225131',
 'mp-1184311',
 'mp-1184260',
 'mp-1224977',
 'mp-1224953',
 'mp-1184444',
 'mp-1224945',
 'mp-1226559',
 'mp-1183837',
 'mp-1008349',
 'mp-

## Listes des pahses stables

In [8]:
print('La base contient {} composés dont {} stables'.format(hull.npoints,len(hull.stable)))
print()
print('Listes des phases stables:')
print('--------------------------')
print()

fmt = '{:2} | {:10} | '+c*'{:6.4f} '+'| {:12.4f}'
fmt_title = '{:>2} | {:^10} | '+c*'x({:2})  '+'| {:^12}'
separation = 3*'-'+'|'+12*'-'+'|'+29*'-'+'|'+14*'-'

print(fmt_title.format('i', 'Nom (MP)',*element, 'E_DFT'))
print(separation)
for i in hull.stable :
    print(fmt.format(i, hull.nom[i], *hull.composition[i], hull.energie[i]))

La base contient 83 composés dont 10 stables

Listes des phases stables:
--------------------------

 i |  Nom (MP)  | x(Fe)  x(Cu)  x(Co)  x(Ni)  |    E_DFT    
---|------------|-----------------------------|--------------
 2 | mp-13      | 1.0000 0.0000 0.0000 0.0000 |      -8.4700
13 | mp-54      | 0.0000 0.0000 1.0000 0.0000 |      -7.1100
21 | mp-23      | 0.0000 0.0000 0.0000 1.0000 |      -5.7800
29 | mp-30      | 0.0000 1.0000 0.0000 0.0000 |      -4.1000
35 | mp-18695   | 0.9375 0.0000 0.0625 0.0000 |      -8.4000
38 | mp-601820  | 0.7500 0.0000 0.2500 0.0000 |      -8.1800
40 | mp-2090    | 0.5000 0.0000 0.5000 0.0000 |      -7.8600
55 | mp-1418    | 0.2500 0.0000 0.0000 0.7500 |      -6.5400
58 | mp-2213    | 0.5000 0.0000 0.0000 0.5000 |      -7.1900
68 | mp-1183837 | 0.0000 0.0000 0.7500 0.2500 |      -6.8000


## Information pour un composé donné

In [11]:
compose='mp-2090'
#
print('Info sur le composé :',compose)
print('------------------------------------')
print('E=',hull.energie_compose(compose))
hull.distance_hull(compose)
pass

Info sur le composé : mp-2090
------------------------------------
E= -7.86
Le composé mp-2090  est stable


In [8]:
compose='mp-1006883'
#
print('Info sur le composé :',compose)
print('------------------------------------')
print('E=',hull.energie_compose(compose))
hull.distance_hull(compose)
pass

Info sur le composé : mp-1006883
------------------------------------
E= -6.29
Le composé mp-1006883  n'est pas stable
--------------------------------------------
ΔH = 0.1700
Il se décompose en :
0.6667(mp-1183837) + 0.3333(mp-23)


## Information pour une composition donnée

In [9]:
composition=[0.2,0.5,0.2,0.1]
#
txt='Compsiton :'
for j in range(hull.dim):
    txt+=' {:5.2f}'.format(composition[j])
print(txt)
print("----------------------------------------")
hull.phases_stables([0.2,0.5,0.2,0.1])
print("----------------------------------------")
print("L'enrgie de cette phase ou de ce mélange de phase est : {:8.4f}".format(hull.energie_hull(composition)[0]))

Compsiton :  0.20  0.50  0.20  0.10
----------------------------------------
Les phases stables sont :
0.3400(mp-2090) + 0.0400(mp-1183837) + 0.1200(mp-1418) + 0.5000(mp-30)
----------------------------------------
L'enrgie de cette phase ou de ce mélange de phase est :  -5.7792


## Ecart à l'enveloppe pour tous les composés

In [11]:
keys, data = hull.tableau_complet(element)
# keys, data = hull.tableau_complet(element, save_csv=True) # Enregistre le tableau dans un format csv
df = pd.DataFrame(data, columns=keys)
df

,Nom,x(Fe),x(Cu),x(Co),x(Ni),Stabilité,Ecart à l'enveloppe
0,mp-1245108,1.000000,0.000000,0.0,0.000000,non,0.28
1,mp-1271198,1.000000,0.000000,0.0,0.000000,non,0.05
2,mp-13,1.000000,0.000000,0.0,0.000000,oui,0.00
3,mp-1194030,1.000000,0.000000,0.0,0.000000,non,0.18
4,mp-1271068,1.000000,0.000000,0.0,0.000000,non,0.02
...,...,...,...,...,...,...,...
78,mp-1225698,0.000000,0.750000,0.0,0.250000,non,0.05
79,mp-1225695,0.000000,0.500000,0.0,0.500000,non,0.07
80,mp-1225687,0.000000,0.500000,0.0,0.500000,non,0.02
81,mp-1184069,0.000000,0.500000,0.0,0.500000,non,0.03
